# 🌿 Crop & Weed Detection with YOLOv8

This notebook trains a YOLOv8 model to detect **crops** and **weeds** in sesame farm images.

### 📋 What this notebook does:
1. Install dependencies
2. Upload & organize your dataset
3. Split into train/val sets
4. Create YOLO config file
5. Train YOLOv8 model
6. Evaluate on validation set
7. Run inference on new images

### 📁 Your data format (already correct!)
Each `.txt` label file uses YOLO format:
```
<class_id> <x_center> <y_center> <width> <height>
```
- `0` = crop
- `1` = weed
- All values are normalized between 0 and 1

> **Runtime:** Make sure `Runtime > Change runtime type > T4 GPU` is selected!

## ⚙️ Step 1: Install Dependencies

In [ ]:
# Install Ultralytics (YOLOv8) — the easiest way to train YOLO models
!pip install ultralytics -q

import os
import shutil
import random
import yaml
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

print('✅ All dependencies installed!')

## 📂 Step 2: Upload Your Dataset

**Option A — Upload a ZIP (recommended):**
- Zip your entire `data/` folder (it should contain images + `.txt` label files together)
- Upload it using the cell below

**Option B — Mount Google Drive:**
- If you've already uploaded to Drive, use the Drive mount cell instead

Run only ONE of the two option cells below.

In [ ]:
# ============================================================
# OPTION A: Upload a ZIP file from your computer
# ============================================================
from google.colab import files
import zipfile

print('📤 Select your data.zip file to upload...')
uploaded = files.upload()  # A file picker will appear

zip_name = list(uploaded.keys())[0]
print(f'\n📦 Extracting {zip_name}...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/raw_data')

print('✅ Upload and extraction complete!')
print('📁 Files found:')
!find /content/raw_data -type f | head -20

In [ ]:
# ============================================================
# OPTION B: Mount Google Drive (skip if you used Option A)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# 👇 Update this path to where your data folder is in your Drive
DRIVE_DATA_PATH = '/content/drive/MyDrive/data'

# Copy to local storage (faster training)
!cp -r "{DRIVE_DATA_PATH}" /content/raw_data
print('✅ Data copied from Google Drive!')

## 🔍 Step 3: Inspect & Understand Your Data

In [ ]:
# ============================================================
# Auto-detect where images and labels are in your uploaded folder
# Handles both: flat structure (images+txts together) and
#               nested structure (images/ and labels/ subfolders)
# ============================================================

RAW_DATA = Path('/content/raw_data')

# Find all image files
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
all_images = [f for f in RAW_DATA.rglob('*') if f.suffix.lower() in IMAGE_EXTS]
all_labels = list(RAW_DATA.rglob('*.txt'))

print(f'📸 Total images found : {len(all_images)}')
print(f'🏷️  Total label files  : {len(all_labels)}')

# Check label contents
class_counts = {0: 0, 1: 0}
for lbl in all_labels:
    with open(lbl) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                cls = int(parts[0])
                class_counts[cls] = class_counts.get(cls, 0) + 1

print(f'\n📊 Bounding box counts:')
print(f'   Class 0 (crop) : {class_counts.get(0, 0)} boxes')
print(f'   Class 1 (weed) : {class_counts.get(1, 0)} boxes')

# Sample one image + label to verify alignment
sample_img = all_images[0]
sample_lbl = sample_img.with_suffix('.txt')
if not sample_lbl.exists():
    # Try sibling labels/ folder
    sample_lbl = sample_img.parent.parent / 'labels' / sample_img.with_suffix('.txt').name

print(f'\n🔎 Sample image : {sample_img.name}')
print(f'🔎 Sample label : {sample_lbl.name if sample_lbl.exists() else "NOT FOUND"}')
if sample_lbl.exists():
    print(f'   Content: {open(sample_lbl).read().strip()}')

In [ ]:
# ============================================================
# Visualize a few images with their bounding boxes
# to confirm labels are correct before training
# ============================================================

CLASS_NAMES = ['crop', 'weed']
CLASS_COLORS = ['#00C853', '#FF1744']  # green=crop, red=weed

def plot_yolo_labels(img_path, lbl_path):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = (xc - bw / 2) * W
                y1 = (yc - bh / 2) * H
                rect = patches.Rectangle(
                    (x1, y1), bw * W, bh * H,
                    linewidth=2, edgecolor=CLASS_COLORS[cls], facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1 - 4, CLASS_NAMES[cls],
                        color=CLASS_COLORS[cls], fontsize=8, fontweight='bold')
    plt.tight_layout()

# Show 4 random samples
samples = random.sample(all_images, min(4, len(all_images)))
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
plt.suptitle('Sample Images with YOLO Labels  (🟢 crop  🔴 weed)', fontsize=13)

for i, img_path in enumerate(samples):
    lbl_path = img_path.with_suffix('.txt')
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    axes[i].imshow(img)
    axes[i].set_title(img_path.name, fontsize=8)
    axes[i].axis('off')
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = (xc - bw / 2) * W
                y1 = (yc - bh / 2) * H
                rect = patches.Rectangle(
                    (x1, y1), bw * W, bh * H,
                    linewidth=2, edgecolor=CLASS_COLORS[cls], facecolor='none'
                )
                axes[i].add_patch(rect)

plt.tight_layout()
plt.show()
print('✅ Labels look correct if boxes surround crops/weeds in the images')

## 🗂️ Step 4: Organize into YOLO Folder Structure

YOLOv8 expects this layout:
```
dataset/
├── images/
│   ├── train/   ← 80% of images
│   └── val/     ← 20% of images
└── labels/
    ├── train/   ← matching .txt files
    └── val/
```

In [ ]:
# ============================================================
# Split data 80% train / 20% val and copy into YOLO structure
# ============================================================

DATASET_DIR = Path('/content/dataset')
TRAIN_RATIO = 0.8  # Change if you want a different split

# Create directories
for split in ['train', 'val']:
    (DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Keep only images that have a matching label file
paired = []
for img in all_images:
    lbl = img.with_suffix('.txt')
    if lbl.exists():
        paired.append((img, lbl))
    else:
        print(f'⚠️  No label found for {img.name} — skipping')

print(f'\n✅ {len(paired)} image-label pairs ready')

# Shuffle and split
random.seed(42)
random.shuffle(paired)
split_idx = int(len(paired) * TRAIN_RATIO)
train_set = paired[:split_idx]
val_set   = paired[split_idx:]

print(f'   Train: {len(train_set)} images')
print(f'   Val  : {len(val_set)} images')

# Copy files
def copy_pairs(pairs, split):
    for img_src, lbl_src in pairs:
        shutil.copy(img_src, DATASET_DIR / 'images' / split / img_src.name)
        shutil.copy(lbl_src, DATASET_DIR / 'labels' / split / lbl_src.name)

copy_pairs(train_set, 'train')
copy_pairs(val_set,   'val')

print('\n📁 Dataset structure created:')
!find /content/dataset -type d

## 📝 Step 5: Create YOLO Config File (`data.yaml`)

In [ ]:
# ============================================================
# YOLO needs a yaml config that tells it:
#  - where the images are
#  - how many classes
#  - what the class names are
# ============================================================

data_yaml = {
    'path'  : str(DATASET_DIR),          # root of dataset
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : 2,                          # number of classes
    'names' : ['crop', 'weed']            # class 0 = crop, class 1 = weed
}

yaml_path = DATASET_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('✅ data.yaml created:')
print(open(yaml_path).read())

## 🚀 Step 6: Train YOLOv8

We use **YOLOv8n** (nano) — smallest and fastest, great for ~1300 images on Colab free tier.

| Model | Size | Speed | Accuracy |
|-------|------|-------|----------|
| yolov8n | 6MB | ⚡⚡⚡ | Good |
| yolov8s | 22MB | ⚡⚡ | Better |
| yolov8m | 50MB | ⚡ | Best (but slower) |

> Change `MODEL = 'yolov8s.pt'` for better accuracy if you have time.

In [ ]:
# ============================================================
# Training configuration — adjust as needed
# ============================================================

MODEL   = 'yolov8n.pt'   # pretrained backbone (auto-downloaded)
EPOCHS  = 50             # increase to 100 for better accuracy
IMGSZ   = 512            # matches your image size (512x512)
BATCH   = 16             # reduce to 8 if you get memory errors

model = YOLO(MODEL)

results = model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = IMGSZ,
    batch   = BATCH,
    name    = 'crop_weed_v1',
    project = '/content/runs',
    device  = 0,           # 0 = GPU, 'cpu' = CPU
    patience= 15,          # early stopping if no improvement for 15 epochs
    save    = True,
    plots   = True,        # saves training curves
    verbose = True
)

print('\n✅ Training complete!')
print(f'📁 Best model saved at: /content/runs/crop_weed_v1/weights/best.pt')

## 📊 Step 7: View Training Results

In [ ]:
# ============================================================
# Show training curves (loss, precision, recall, mAP)
# ============================================================

results_dir = Path('/content/runs/crop_weed_v1')

# Display the auto-generated results plot
results_img = results_dir / 'results.png'
if results_img.exists():
    img = Image.open(results_img)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Metrics', fontsize=14)
    plt.tight_layout()
    plt.show()

# Show confusion matrix
conf_matrix = results_dir / 'confusion_matrix_normalized.png'
if conf_matrix.exists():
    img = Image.open(conf_matrix)
    plt.figure(figsize=(7, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Run validation on the val set to get mAP50 score
# ============================================================

best_model_path = '/content/runs/crop_weed_v1/weights/best.pt'
best_model = YOLO(best_model_path)

val_results = best_model.val(
    data   = str(yaml_path),
    imgsz  = IMGSZ,
    device = 0
)

print('\n📈 Validation Results:')
print(f'   mAP50       : {val_results.box.map50:.4f}')    # main metric
print(f'   mAP50-95    : {val_results.box.map:.4f}')       # stricter metric
print(f'   Precision   : {val_results.box.mp:.4f}')
print(f'   Recall      : {val_results.box.mr:.4f}')
print('\n💡 mAP50 > 0.7 is good. > 0.85 is great for this dataset size.')

## 🔍 Step 8: Run Inference on New Images

Upload images you want to detect crops/weeds in.

In [ ]:
# ============================================================
# Upload new images for inference
# ============================================================

from google.colab import files

print('📤 Upload images to run inference on...')
uploaded_infer = files.upload()

# Save to a folder
infer_dir = Path('/content/inference_input')
infer_dir.mkdir(exist_ok=True)

for fname, fdata in uploaded_infer.items():
    with open(infer_dir / fname, 'wb') as f:
        f.write(fdata)

print(f'✅ {len(uploaded_infer)} image(s) ready for detection')

In [ ]:
# ============================================================
# Run detection and show results
# ============================================================

CONF_THRESHOLD = 0.25   # minimum confidence to show a detection (0.0 - 1.0)
IOU_THRESHOLD  = 0.45   # NMS threshold (lower = fewer overlapping boxes)

# Run inference
predictions = best_model.predict(
    source = str(infer_dir),
    conf   = CONF_THRESHOLD,
    iou    = IOU_THRESHOLD,
    imgsz  = IMGSZ,
    device = 0,
    save   = True,
    project= '/content/runs',
    name   = 'inference'
)

# Display results inline
CLASS_COLORS_BGR = {'crop': '#00C853', 'weed': '#FF1744'}

fig_cols = min(len(predictions), 3)
fig, axes = plt.subplots(1, fig_cols, figsize=(7 * fig_cols, 7))
if fig_cols == 1:
    axes = [axes]

for i, result in enumerate(predictions[:3]):
    # YOLOv8 saves annotated image — load it
    annotated = result.plot()  # returns BGR numpy array
    annotated_rgb = annotated[:, :, ::-1]  # BGR → RGB
    axes[i].imshow(annotated_rgb)
    axes[i].axis('off')

    # Count detections per class
    n_crop = sum(1 for c in result.boxes.cls.tolist() if int(c) == 0)
    n_weed = sum(1 for c in result.boxes.cls.tolist() if int(c) == 1)
    axes[i].set_title(
        f'{Path(result.path).name}\n🌱 {n_crop} crop  🌿 {n_weed} weed',
        fontsize=10
    )

plt.suptitle('Detection Results', fontsize=14)
plt.tight_layout()
plt.show()

print('\n📋 Summary:')
for result in predictions:
    n_crop = sum(1 for c in result.boxes.cls.tolist() if int(c) == 0)
    n_weed = sum(1 for c in result.boxes.cls.tolist() if int(c) == 1)
    confs  = result.boxes.conf.tolist()
    avg_conf = sum(confs) / len(confs) if confs else 0
    print(f'  {Path(result.path).name}: {n_crop} crops, {n_weed} weeds  (avg conf: {avg_conf:.2f})')

## 💾 Step 9: Download Your Trained Model

In [ ]:
# ============================================================
# Download the best trained model weights to your computer
# ============================================================

from google.colab import files

print('📥 Downloading best.pt to your computer...')
files.download('/content/runs/crop_weed_v1/weights/best.pt')
print('✅ Done! Save this file — it is your trained model.')
print('\n💡 To use it later: model = YOLO("best.pt")')

## 🔁 Step 10: (Optional) Reload Saved Model & Run Inference Later

If you close the notebook and come back later, run this cell (after re-uploading your model).

In [ ]:
# ============================================================
# Load a previously saved model and run inference
# (use this in a fresh session after re-uploading best.pt)
# ============================================================

# from google.colab import files
# uploaded_model = files.upload()   # upload best.pt
# saved_model = YOLO('best.pt')

# saved_model.predict(
#     source = 'path/to/image.jpg',
#     conf   = 0.25,
#     show   = True
# )

print('Uncomment and run the lines above to load and use a saved model.')